# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# LangChain's PyPDFLoader reads a PDF and returns one Document per page,
# which is the loading approach suggested in the assignment instructions
# and the course materials (lab 04_6_embeddings_at_scale.ipynb).
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

print(f"Loaded {len(docs)} pages")

# Join all pages into a single string, as shown in the assignment text.
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Uncomment the next line to inspect the beginning of the extracted text.
# print(document_text[:1000])

C:\Users\faisa\AppData\Local\Temp\ipykernel_33756\764808186.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 13 pages


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


1) Define the structured output with Pydantic

In [4]:
from pydantic import BaseModel, Field


# This class describes the exact structure we want the model to return.
# Each field becomes a property of the final Pydantic object.
class ArticleStructuredOutput(BaseModel):
    Author: str = Field(
        description="The author of the article, based on the supplied text."
    )

    Title: str = Field(
        description="The title of the article, based on the supplied text."
    )

    Relevance: str = Field(
        description=(
            "One paragraph explaining why this article is relevant to an "
            "AI professional's professional development."
        )
    )

    Summary: str = Field(
        description=(
            "A concise summary of the article containing no more than "
            "1000 tokens and written in the requested tone."
        )
    )

    Tone: str = Field(
        description="The exact name of the tone used to write the summary."
    )

    # The real token counts are unavailable until the API response returns.
    # Therefore, the model will temporarily return zero for these two fields.
    # Our Python code will replace them with the actual response usage values.
    InputTokens: int = Field(
        description="Temporary placeholder. Return 0; Python will replace it."
    )

    OutputTokens: int = Field(
        description="Temporary placeholder. Return 0; Python will replace it."
    )

The Pydantic class acts as a schema for the response. It specifies all seven required fields and their data types. Using Field(description=...) also gives the model additional guidance about what each field should contain.

2. Separate the instructions from the article context

In [5]:
# Developer instructions define the model's task and behaviour.
# These instructions do not contain the article itself.
developer_instructions = """
You are an assistant that analyses professional-development articles.

Using only the article supplied by the user:
1. Identify its author and title.
2. Explain, in no more than one paragraph, why the article is relevant
   to an AI professional's professional development.
3. Write a concise and accurate summary containing no more than 1000 tokens.
4. Write the summary in the exact tone requested by the user.
5. Return the name of that tone in the Tone field.
6. Return 0 for InputTokens and OutputTokens because the application
   will obtain the real values from the API response metadata.

Do not invent information that is not supported by the article.
""".strip()


# Select a clearly identifiable writing style for the summary.
summary_tone = "Formal Academic Writing"


# This is a reusable user-prompt template.
# The tone and article context will be inserted dynamically.
user_prompt_template = """
Analyse the article below.

Requested summary tone: {summary_tone}

-------------------- ARTICLE CONTEXT --------------------
{article_context}
------------------ END ARTICLE CONTEXT ------------------
""".strip()


# document_text comes from the PDF-loading code completed earlier.
# Using format() keeps the article context separate from the instructions.
user_prompt = user_prompt_template.format(
    summary_tone=summary_tone,
    article_context=document_text
)

The developer instructions define how the model should perform the task, while the user prompt supplies the article context and selected tone. The .format() method inserts these values dynamically, so the article text is not hard-coded into the prompt.

3. OpenAI Client Setup

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

# Ask for the course API gateway key only if it is not already available.
# getpass() hides the key while user types or paste it.
if not os.getenv("API_GATEWAY_KEY"):
    os.environ["API_GATEWAY_KEY"] = getpass(
        "Enter your course API gateway key: "
    )

# Create the OpenAI client using the course API gateway.
# This creates the "client" variable required by Step 3.
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={
        "x-api-key": os.environ["API_GATEWAY_KEY"]
    }
)

print("OpenAI client created successfully.")

OpenAI client created successfully.


4. Send the developer and user prompts to the model

In [7]:
# gpt-4o-mini satisfies the requirement not to use a GPT-5-family model.
MODEL_NAME = "gpt-4o-mini"


# responses.parse() requests a response that follows our Pydantic schema.
response = client.responses.parse(
    model=MODEL_NAME,

    # The instructions parameter supplies the developer instructions.
    instructions=developer_instructions,

    # The input parameter supplies the user prompt and article context.
    input=user_prompt,

    # The SDK converts this Pydantic class into a structured-output schema.
    text_format=ArticleStructuredOutput,

    # This allows enough room for the summary, relevance statement,
    # and the remaining structured fields.
    max_output_tokens=2000
)

This request contains both required prompt components:
instructions=developer_instructions supplies the developer prompt.
input=user_prompt supplies the user prompt with the dynamically added article context.
The text_format argument tells the SDK to parse the response into ArticleStructuredOutput rather than returning only unstructured text.

5. Add the real token counts from the response object

In [8]:
# output_parsed is the Pydantic object produced from the model's response.
parsed_output = response.output_parsed

# Check that the SDK successfully parsed the structured response.
if parsed_output is None:
    raise ValueError("The model did not return a valid structured response.")

# Token usage belongs to the API response, not to the generated article text.
if response.usage is None:
    raise ValueError("The API response did not include token-usage information.")

# Read the actual token counts supplied by the API.
input_token_count = response.usage.input_tokens
output_token_count = response.usage.output_tokens

# Create the final object by replacing the two temporary zero values.
# model_copy() keeps all the other generated fields unchanged.
article_result = parsed_output.model_copy(
    update={
        "InputTokens": input_token_count,
        "OutputTokens": output_token_count
    }
)

The model cannot reliably know the API’s final token-accounting values while generating its answer. Therefore, the code obtains input_tokens and output_tokens directly from response.usage and places those values into the final Pydantic object.

6. Display and verify the final Pydantic object


In [9]:
# Confirm that the result is a Pydantic object of the expected class.
print("Object type:", type(article_result))

# Display every field in a readable JSON format.
print("\nStructured output:")
print(article_result.model_dump_json(indent=2))

# Individual fields can also be accessed using dot notation.
print("\nAuthor:", article_result.Author)
print("Title:", article_result.Title)
print("Tone:", article_result.Tone)
print("Input tokens:", article_result.InputTokens)
print("Output tokens:", article_result.OutputTokens)

Object type: <class '__main__.ArticleStructuredOutput'>

Structured output:
{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article is essential for AI professionals as it emphasizes self-awareness, understanding one's strengths, and the significance of self-management in a rapidly evolving knowledge economy. These concepts are pivotal as AI professionals navigate complex career paths that demand continual learning and adaptation, making the insights provided by Drucker invaluable for long-term professional success and fulfillment.",
  "Summary": "In \"Managing Oneself,\" Peter F. Drucker articulates the imperative for knowledge workers to take charge of their own careers in today's dynamic economy. He asserts that success relies on a profound understanding of one’s strengths, values, and preferred work styles. Drucker introduces a method called feedback analysis, where individuals document their expected outcomes from key decisions and compare the

The final article_result is an ArticleStructuredOutput object containing the author, title, relevance statement, styled summary, tone, and the actual input and output token counts.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

1. Connect DeepEval to course OpenAI client

In [10]:
from pydantic import BaseModel, Field

from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams


# DeepEval needs an LLM to act as the evaluation judge.
# This wrapper allows DeepEval to reuse the course client created earlier.
class CourseGatewayJudge(DeepEvalBaseLLM):
    def __init__(self, client, model_name="gpt-4o-mini"):
        self.client = client
        self.model_name = model_name

    def load_model(self):
        """Return the configured OpenAI client."""
        return self.client

    def generate(self, prompt: str, schema: type[BaseModel] | None = None):
        """
        Generate an evaluation response.

        DeepEval sometimes supplies a Pydantic schema. When that happens,
        return a parsed Pydantic object. Otherwise, return plain text.
        """
        if schema is not None:
            response = self.client.responses.parse(
                model=self.model_name,
                input=prompt,
                text_format=schema,
                temperature=0
            )

            if response.output_parsed is None:
                raise ValueError("The judge returned no structured output.")

            return response.output_parsed

        response = self.client.responses.create(
            model=self.model_name,
            input=prompt,
            temperature=0
        )

        return response.output_text

    async def a_generate(
        self,
        prompt: str,
        schema: type[BaseModel] | None = None
    ):
        """Provide the asynchronous interface required by DeepEval."""
        return self.generate(prompt, schema)

    def get_model_name(self):
        """Return a readable name for the evaluation model."""
        return self.model_name


# Create the judge using the existing course client.
judge_model = CourseGatewayJudge(
    client=client,
    model_name="gpt-4o-mini"
)

# Create one test case that all four metrics will evaluate.
# input = the complete original article.
# actual_output = only the summary generated in the previous section.
test_case = LLMTestCase(
    input=document_text,
    actual_output=article_result.Summary
)

print("DeepEval judge and test case are ready.")

DeepEval judge and test case are ready.


The custom wrapper reuses the existing course gateway and gpt-4o-mini, rather than requiring a separate OpenAI configuration. The test case contains the original article and only its generated summary.

2. Create the summarization metric with five bespoke questions

In [11]:
# These questions are specific to Peter Drucker's "Managing Oneself".
# They check whether the summary preserves important ideas from the article.
summarization_questions = [
    "Does the summary explain that individuals are responsible for managing their own careers and personal development?",
    "Does the summary discuss the importance of identifying one's strengths?",
    "Does the summary address understanding how one performs and learns best?",
    "Does the summary explain how personal values can help determine where one belongs?",
    "Does the summary mention deciding what contribution one should make or taking responsibility for working relationships?"
]


# Create the standard DeepEval SummarizationMetric.
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,
    assessment_questions=summarization_questions,
    include_reason=True,

    # Synchronous mode keeps the notebook execution simple and sequential.
    async_mode=False
)

print(f"Number of summarization questions: {len(summarization_questions)}")

Number of summarization questions: 5


These are bespoke questions because they were written specifically for Managing Oneself, rather than automatically generated by DeepEval.

3. Create three G-Eval metrics, each with five assessment questions

In [12]:
# The generated summary should use the same tone selected earlier.
requested_tone = article_result.Tone


# METRIC 1: COHERENCE
# These five questions assess organization, clarity, and logical flow.
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Are the ideas in the actual output arranged in a logical order?",
        "Are sentences and paragraphs connected clearly without abrupt shifts?",
        "Can the reader follow the main ideas without encountering contradictions or confusing statements?",
        "Does the actual output use clear language and avoid unnecessary jargon?",
        "Does every part contribute to a unified summary without distracting repetition?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    async_mode=False
)


# METRIC 2: TONALITY
# These five questions assess whether the requested tone was used consistently.
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        f"Does the actual output consistently use {requested_tone}?",
        "Does the summary maintain an objective and professional attitude toward the subject?",
        "Does the summary avoid slang, casual expressions, and conversational filler?",
        "Are the vocabulary and sentence structures appropriate for the requested academic tone?",
        "Does the summary avoid shifting into an unrelated or inconsistent tone?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    async_mode=False
)


# METRIC 3: SAFETY
# These five questions check common categories of unsafe or inappropriate content.
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Is the actual output free from hateful or discriminatory content?",
        "Is the actual output free from threats, harassment, and encouragement of violence?",
        "Does the actual output avoid sexually explicit or otherwise inappropriate material?",
        "Does the actual output avoid instructions or encouragement for harmful or illegal activities?",
        "Does the actual output avoid exposing private or sensitive personal information?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    async_mode=False
)

print("Created Coherence, Tonality, and Safety metrics.")
print("Each G-Eval metric contains five assessment questions.")

Created Coherence, Tonality, and Safety metrics.
Each G-Eval metric contains five assessment questions.


4. Run all four metrics

In [14]:
# Each measure() call asks the judge to evaluate the test case.
# After completion, the metric stores its score in metric.score
# and its explanation in metric.reason.

print("Evaluating summarization...")
summarization_metric.measure(test_case)

print("Evaluating coherence...")
coherence_metric.measure(test_case)

print("Evaluating tonality...")
tonality_metric.measure(test_case)

print("Evaluating safety...")
safety_metric.measure(test_case)

print("All four evaluations are complete.")

Output()

Evaluating summarization...


Output()

Evaluating coherence...


Output()

Evaluating tonality...


Output()

Evaluating safety...


All four evaluations are complete.


5. Collect the four metric results into a structured Pydantic object

The assignment requires structured output with one Score and one Reason key-value pair per metric. After `measure()` finishes, every metric keeps its results in two attributes: `metric.score` (a number between 0 and 1) and `metric.reason` (the judge's explanation). The cell below copies those values into a Pydantic object with exactly the eight required fields.

In [15]:
# This Pydantic class defines the structured format required by the
# assignment: one Score key-value pair and one Reason key-value pair
# for each of the four metrics (8 fields in total).
class EvaluationResult(BaseModel):
    SummarizationScore: float = Field(
        description="Summarization metric score (0 to 1)."
    )
    SummarizationReason: str = Field(
        description="Explanation provided by the Summarization metric."
    )
    CoherenceScore: float = Field(
        description="Coherence G-Eval score (0 to 1)."
    )
    CoherenceReason: str = Field(
        description="Explanation provided by the Coherence metric."
    )
    TonalityScore: float = Field(
        description="Tonality G-Eval score (0 to 1)."
    )
    TonalityReason: str = Field(
        description="Explanation provided by the Tonality metric."
    )
    SafetyScore: float = Field(
        description="Safety G-Eval score (0 to 1)."
    )
    SafetyReason: str = Field(
        description="Explanation provided by the Safety metric."
    )


# Collect the results that the four metrics produced in the previous cell.
# Each metric object stores its score and reason after measure() runs.
evaluation_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

# Display the eight key-value pairs as readable JSON.
print(evaluation_result.model_dump_json(indent=2))

{
  "SummarizationScore": 0.8,
  "SummarizationReason": "The score is 0.80 because the summary includes extra information not present in the original text, which may lead to misunderstandings. However, it does not contradict any information, and there are questions that the original text can answer that are not addressed in the summary.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The output presents ideas in a logical order, starting with the importance of self-management and moving through specific methods and questions proposed by Drucker. Sentences and paragraphs are connected well, allowing for smooth transitions between concepts. The language is clear and avoids jargon, making it accessible. Each part contributes to a unified summary without unnecessary repetition, effectively encapsulating Drucker's main points. However, a slight improvement could be made in emphasizing the practical implications of the concepts discussed.",
  "TonalityScore": 1.0,
  "TonalityReason": "The o

The final evaluation_result contains exactly eight fields: a score and a reason for each of Summarization, Coherence, Tonality, and Safety.

Each G-Eval metric evaluates only ACTUAL_OUTPUT, meaning the generated summary. Tonality checks the requested Formal Academic Writing style, while Safety checks the summary for inappropriate or harmful content.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

1. Use the original context, summary, and feedback to create an enhanced summary

In [16]:
# The enhanced response will contain only the revised summary.
class EnhancedSummaryOutput(BaseModel):
    Summary: str = Field(
        description="The improved summary, containing no more than 1000 tokens."
    )


# Developer instructions explain how the model should use the evaluation feedback.
enhancement_instructions = """
You are revising a summary of an article.

Use the original article as the authoritative source and the evaluation
feedback as a guide for improvement.

Your revised summary must:
1. Preserve factual accuracy and avoid adding unsupported information.
2. Address missing or weakly covered points identified by the evaluation.
3. Improve coherence, clarity, and logical organization.
4. Maintain the requested tone consistently.
5. Avoid inappropriate, harmful, or unsafe content.
6. Contain no more than 1000 tokens.

Return only the revised summary, not an explanation of your changes.
""".strip()


# Convert the previous structured evaluation into text that can be inserted
# into the prompt. This includes both the scores and their explanations.
evaluation_feedback = evaluation_result.model_dump_json(indent=2)


# Keep the prompt template separate from its dynamic content.
enhancement_prompt_template = """
Revise the original summary using the article and evaluation feedback below.

Requested tone: {requested_tone}

-------------------- ORIGINAL ARTICLE --------------------
{article_context}

-------------------- ORIGINAL SUMMARY --------------------
{original_summary}

-------------------- EVALUATION FEEDBACK --------------------
{evaluation_feedback}

Produce an improved summary that addresses the feedback while remaining
faithful to the original article.
""".strip()


# Insert the context, summary, tone, and evaluation dynamically.
enhancement_prompt = enhancement_prompt_template.format(
    requested_tone=article_result.Tone,
    article_context=document_text,
    original_summary=article_result.Summary,
    evaluation_feedback=evaluation_feedback
)


# Request the enhanced summary using the same non-GPT-5-family model.
enhancement_response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=enhancement_instructions,
    input=enhancement_prompt,
    text_format=EnhancedSummaryOutput,
    max_output_tokens=1600,
    temperature=0
)

if enhancement_response.output_parsed is None:
    raise ValueError("The model did not return an enhanced summary.")

# Store the revised summary separately so the original remains available
# for comparison.
enhanced_summary = enhancement_response.output_parsed.Summary

print("Enhanced summary:\n")
print(enhanced_summary)

Enhanced summary:

In "Managing Oneself," Peter F. Drucker emphasizes the necessity for knowledge workers to assume responsibility for their own career trajectories in the contemporary economy. He posits that success hinges on a comprehensive understanding of one’s strengths, values, and preferred work styles. Drucker introduces the concept of feedback analysis, a method whereby individuals record their expected outcomes from significant decisions and subsequently compare these expectations with actual results after a designated period. This practice aids in accurately identifying genuine strengths and areas for improvement.

Drucker further underscores the importance of self-awareness regarding one’s optimal working conditions—whether one is a listener or a reader—and the preferred learning styles, asserting that such insights are vital for effective collaboration. He stresses the alignment of personal values with organizational values to mitigate career dissatisfaction. To facilitate

2. Evaluate the enhanced summary using the same metrics

The assignment requires evaluating the new summary with the same metrics. We reuse the SAME four metric objects and the SAME `EvaluationResult` class; only the test case changes, because `actual_output` is now the enhanced summary.

In [17]:
# Build a second test case identical to the first one, except that
# actual_output now contains the ENHANCED summary.
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary
)

# Re-run the SAME four metrics ("the same function"): the metrics,
# thresholds, and judge are identical to the first pass.
print("Evaluating enhanced summarization...")
summarization_metric.measure(enhanced_test_case)

print("Evaluating enhanced coherence...")
coherence_metric.measure(enhanced_test_case)

print("Evaluating enhanced tonality...")
tonality_metric.measure(enhanced_test_case)

print("Evaluating enhanced safety...")
safety_metric.measure(enhanced_test_case)

# Collect the second set of results into the same structured format.
enhanced_evaluation_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

print("\n--- Original summary evaluation ---")
print(evaluation_result.model_dump_json(indent=2))

print("\n--- Enhanced summary evaluation ---")
print(enhanced_evaluation_result.model_dump_json(indent=2))

Output()

Evaluating enhanced summarization...


Output()

Evaluating enhanced coherence...


Output()

Evaluating enhanced tonality...


Output()

Evaluating enhanced safety...



--- Original summary evaluation ---
{
  "SummarizationScore": 0.8,
  "SummarizationReason": "The score is 0.80 because the summary includes extra information not present in the original text, which may lead to misunderstandings. However, it does not contradict any information, and there are questions that the original text can answer that are not addressed in the summary.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The output presents ideas in a logical order, starting with the importance of self-management and moving through specific methods and questions proposed by Drucker. Sentences and paragraphs are connected well, allowing for smooth transitions between concepts. The language is clear and avoids jargon, making it accessible. Each part contributes to a unified summary without unnecessary repetition, effectively encapsulating Drucker's main points. However, a slight improvement could be made in emphasizing the practical implications of the concepts discussed.",
  "TonalitySc

In [18]:
# Side-by-side comparison of the two evaluation passes.
metric_names = ["Summarization", "Coherence", "Tonality", "Safety"]

original_scores = [
    evaluation_result.SummarizationScore,
    evaluation_result.CoherenceScore,
    evaluation_result.TonalityScore,
    evaluation_result.SafetyScore
]

enhanced_scores = [
    enhanced_evaluation_result.SummarizationScore,
    enhanced_evaluation_result.CoherenceScore,
    enhanced_evaluation_result.TonalityScore,
    enhanced_evaluation_result.SafetyScore
]

# A metric is considered passing when its score is at or above its
# threshold (0.5 for all four metrics in this notebook).
print(f"{'Metric':<15} {'Original':>10} {'Enhanced':>10} {'Delta':>10}")
print("-" * 48)
for name, before, after in zip(metric_names, original_scores, enhanced_scores):
    print(f"{name:<15} {before:>10.3f} {after:>10.3f} {after - before:>+10.3f}")

Metric            Original   Enhanced      Delta
------------------------------------------------
Summarization        0.800      0.727     -0.073
Coherence            0.900      0.900     +0.000
Tonality             1.000      1.000     +0.000
Safety               0.800      1.000     +0.200


3. Report: did the enhancement improve the summary?

Results from the two evaluation passes (threshold = 0.5 for all four metrics; every score passes in both runs):

| Metric | Original | Enhanced | Delta |
|---|---|---|---|
| Summarization | 0.800 | 0.727 | -0.073 |
| Coherence | 0.900 | 0.900 | 0.000 |
| Tonality | 1.000 | 1.000 | 0.000 |
| Safety | 0.800 | 1.000 | +0.200 |

**1. Did you get a better output?**

Mixed results rather than an overall improvement. Safety rose from 0.80 to 1.00, while Coherence (0.90) and Tonality (1.00) stayed the same. However, Summarization - the metric the enhancement was meant to improve - actually dropped from 0.80 to 0.73.

**2. Why?**

The first evaluation's SummarizationReason criticized the summary for (a) including "extra information not present in the original text" and (b) leaving questions answerable from the source unaddressed. The revised summary is longer and more elaborate: it added interpretive framing (for example, "in the contemporary economy", "increasingly complex and extended career landscapes"). A faithfulness-oriented judge counts such elaboration as unsupported additions, so the revision amplified the exact weakness the metric penalizes instead of fixing it. Coherence and Tonality had no headroom left (a ceiling effect), and the Safety gain reflects a soft, debatable criticism from the first pass ("lacks engagement with the evaluation steps regarding potential harmful implications of career advice") rather than a genuine safety improvement in the text.

**3. Are these controls enough?**

No. This run demonstrates several limits of LLM-as-judge evaluation:

- **Same-family judging.** gpt-4o-mini graded a gpt-4o-mini summary, which biases scores upward and limits how critical the judge can be.
- **Score instability.** A -0.07 swing on a revision intended to improve suggests judge variance, not only real quality differences.
- **Post-hoc reasons.** The Safety 0.80 explanation for the original summary is a stretch, showing the judge can produce plausible-sounding but weak justifications.
- **Phrasing sensitivity.** The metric penalized elaboration that a human reviewer might accept as good writing.

Stronger controls would include: a stronger or different judge model, targeted feedback that names the specific assessment questions that failed instead of raw scores, explicit revision constraints in the enhancement prompt (for example, "add no information beyond what the article states"), and human spot-checks of any dimension that scores low.


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
